In [ ]:
import os
import numpy as np

def get_porosity_stats(folder_path, dimensions=(128, 128, 128), pore_value=1):
    """
    Calculate porosity for all RAW files in a directory and return four summary statistics.
    """
    if not os.path.exists(folder_path):
        return None  # Return None if the directory does not exist

    porosity_list = []
    expected_size = dimensions[0] * dimensions[1] * dimensions[2]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.raw'):
            file_path = os.path.join(folder_path, filename)
            
            try:
                data = np.fromfile(file_path, dtype=np.uint8)
                if data.size != expected_size:
                    print(f"  [Skip] {filename}: size mismatch (expected {expected_size}, found {data.size})")
                    continue # Skip files with incompatible sizes
                
                # Calculate porosity for this file
                pore_voxels = np.count_nonzero(data == pore_value)
                porosity = pore_voxels / data.size
                porosity_list.append(porosity)
                
                # ====== Report porosity for each file ======
                print(f"  - File: {filename:<20} | Porosity: {porosity:.4f}")
                
            except Exception as e:
                print(f"  [Error] Cannot read {filename}: {e}")
                pass # Continue after a read error

    if not porosity_list:
        return None

    # Calculate mean, minimum, maximum and standard deviation
    return {
        "Mean": np.mean(porosity_list),
        "Minimum": np.min(porosity_list),
        "Maximum": np.max(porosity_list),
        "Standard deviation": np.std(porosity_list) # Population standard deviation
    }

def print_formatted_table(results_dict):
    """
    Format the summary statistics as a table.
    """
    # Exclude models with invalid input paths
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    
    if not valid_results:
        print("\nNo valid data found; check the directory paths.")
        return

    models = list(valid_results.keys())
    stats_keys = ["Mean", "Minimum", "Maximum", "Standard deviation"]
    
    print("\n" + "="*50 + " Final statistical summary " + "="*50)
    # Print the table header
    print("-" * 75)
    print(f"{'Porosity':<22} TI_general and reconstructions")
    print("-" * 75)
    
    # Print column names
    header_format = "{:<22}" + "{:>12}" * len(models)
    print(header_format.format("", *models))
    print("-" * 75)
    
    # Print rows with four decimal places
    for stat in stats_keys:
        row_values = [stat]
        for model in models:
            val = valid_results[model][stat]
            row_values.append(f"{val:.6f}")
        print(header_format.format(*row_values))
    
    print("-" * 75)

if __name__ == "__main__":
    # ================= Configuration =================
    # 1. Set the input directory paths here.
    # Dictionary keys become table column names; values are input directory paths.
    TARGET_FOLDERS = {
        "TI_general": r"../../data/ablations/wo-dit-b",       # Set the TI_general directory
#         "DCGAN": r"../../data/baselines/dcgan",     # Set the DCGAN directory (optional)
#         "StyleGAN": r"../../data/baselines/stylegan", # Set the StyleGAN directory (optional)
#         "STGAN": r"../../data/baselines/stgan"      # Set the STGAN directory (optional)
    }

    # 2. Volume dimensions
    VOLUME_DIMS = (128, 128, 128)
    
    # 3. Pore-phase value
    PORE_VAL = 1  
    # ==========================================

    print("Scanning files and calculating porosity...\n")
    
    all_results = {}
    for model_name, folder in TARGET_FOLDERS.items():
        print(f"Processing model [{model_name}] (directory: {folder})...")
        # Invalid directory paths are skipped
        stats = get_porosity_stats(folder, VOLUME_DIMS, PORE_VAL)
        if stats:
            all_results[model_name] = stats
        else:
            print(f"  [Warning] No valid RAW volumes found in {folder}.\n")
        print("") # Separate model outputs with a blank line
            
    # Print the final table
    print_formatted_table(all_results)

In [ ]:
import os
import numpy as np
from skimage.filters import threshold_otsu

def process_and_get_stats(folder_path, dimensions=(128, 128, 128)):
    """
    Process RAW files in a directory using Otsu thresholding,
    calculate porosity and save outputs in the 'binarized_results' subdirectory.
    """
    if not os.path.exists(folder_path):
        print(f"Directory does not exist: {folder_path}")
        return None

    # --- Create the output subdirectory ---
    output_subfolder = os.path.join(folder_path, "binarized_results")
    if not os.path.exists(output_subfolder):
        os.makedirs(output_subfolder)
        print(f"Created output subdirectory: {output_subfolder}")

    porosity_list = []
    expected_size = dimensions[0] * dimensions[1] * dimensions[2]

    # Collect input files
    raw_files = [f for f in os.listdir(folder_path) 
                 if f.lower().endswith('.raw')]
    
    if not raw_files:
        print("No RAW files found for processing.")
        return None

    for filename in raw_files:
        file_path = os.path.join(folder_path, filename)
        
        try:
            # 1. Read data as int8
            data = np.fromfile(file_path, dtype=np.uint8)
            if data.size != expected_size:
                print(f"  [Skip] {filename}: size mismatch (expected {expected_size}, found {data.size})")
                continue
            
            # 2. Apply Otsu's automatic thresholding
            try:
                thresh = threshold_otsu(data)
            except ValueError:
                print(f"  [Warning] {filename}: constant intensity; cannot compute an automatic threshold. Skipped.")
                continue

            # 3. Binarize the volume
            # The original '>' convention is retained. For dark pores, change the comparison to '<'.
            binary_data = np.where(data > thresh, 1, 0).astype(np.int8)

            # 4. Calculate porosity
            pore_voxels = np.count_nonzero(binary_data == 1)
            porosity = pore_voxels / data.size
            porosity_list.append(porosity)
            
            # 5. Save to the output subdirectory
            out_filename = filename[:-4] + "_binarized.raw"
            out_path = os.path.join(output_subfolder, out_filename) # Set the output path here
            binary_data.tofile(out_path)
            
            print(f"  - File: {filename:<20} | Threshold: {thresh:>4} | Porosity: {porosity:.4f} | Saved: binarized_results/{out_filename}")
            
        except Exception as e:
            print(f"  [Error] Failed to process {filename}: {e}")

    if not porosity_list:
        return None

    return {
        "Mean": np.mean(porosity_list),
        "Minimum": np.min(porosity_list),
        "Maximum": np.max(porosity_list),
        "Standard deviation": np.std(porosity_list)
    }

def print_single_folder_stats(stats_dict, folder_name):
    if not stats_dict:
        print("\nNo valid statistics were generated.")
        return

    print("\n" + "="*50)
    print(f" Final statistical summary: {folder_name}")
    print("="*50)
    print("-" * 45)
    print(f"{'Statistic':<25} {'Value':>15}")
    print("-" * 45)
    
    stats_keys = ["Mean", "Minimum", "Maximum", "Standard deviation"]
    for stat in stats_keys:
        val = stats_dict[stat]
        print(f"{stat:<25} {val:>15.4f}")
    print("-" * 45)

if __name__ == "__main__":
    # ================= Configuration =================
    TARGET_FOLDER = r"../../data/baselines/stgan"  
    VOLUME_DIMS = (128, 128, 128)
    # ==========================================

    print("Scanning files, applying automatic thresholding and calculating porosity...\n")
    results = process_and_get_stats(TARGET_FOLDER, VOLUME_DIMS)
    print_single_folder_stats(results, TARGET_FOLDER)